# ONS dataset exploration

This notebook downloads the ONS datasets relevant to the housing intelligence project, checks the available files, and profiles each dataset for schema, nulls, and categorical uniqueness.


In [1]:
import os
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests

ONS_BASE = "https://api.beta.ons.gov.uk/v1"
DATASET_IDS = [
    "house-prices-local-authority",
    "index-private-housing-rental-prices",
    "mid-year-pop-est",
    "ageing-population-estimates",
    "projections-older-people-in-single-households",
    "older-people-net-internal-migration",
    "ashe-tables-7-and-8",
    "labour-market",
    "wellbeing-local-authority",
    "life-expectancy-by-local-authority",
    "gdp-by-local-authority",
    "regional-gdp-by-year",
    "regional-gdp-by-quarter",
    "output-in-the-construction-industry",
    "wellbeing-quarterly",
    "suicides-in-the-uk",
    "ageing-population-projections",
    "older-people-economic-activity",
]

NOTEBOOK_DIR = Path.cwd()
LOCAL_DATA_DIR = NOTEBOOK_DIR / "datasets"
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR = NOTEBOOK_DIR / "profiles"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import json
import os
import time
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests

ONS_BASE = "https://api.beta.ons.gov.uk/v1"
DATASET_IDS = [
    "house-prices-local-authority",
    "index-private-housing-rental-prices",
    "mid-year-pop-est",
    "ageing-population-estimates",
    "projections-older-people-in-single-households",
    "older-people-net-internal-migration",
    "ashe-tables-7-and-8",
    "labour-market",
    "wellbeing-local-authority",
    "life-expectancy-by-local-authority",
    "gdp-by-local-authority",
    "regional-gdp-by-year",
    "regional-gdp-by-quarter",
    "output-in-the-construction-industry",
    "wellbeing-quarterly",
    "suicides-in-the-uk",
    "ageing-population-projections",
    "older-people-economic-activity",
]

NOTEBOOK_DIR = Path.cwd()
LOCAL_DATA_DIR = NOTEBOOK_DIR / "datasets"
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR = NOTEBOOK_DIR / "profiles"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)


def get_json(url: str, params: Optional[dict[str, Any]] = None) -> Any:
    time.sleep(0.5)
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


def get_latest_version_url(dataset_id: str) -> str:
    dataset_meta = get_json(f"{ONS_BASE}/datasets/{dataset_id}")
    latest = dataset_meta.get("links", {}).get("latest_version")
    if not latest:
        raise ValueError(f"No latest_version found for dataset: {dataset_id}")
    return latest["href"]


def get_download_url(version_url: str) -> tuple[str, str]:
    version_data = get_json(version_url)
    downloads = version_data.get("downloads", {})

    preferred_order = ["csv", "xlsx", "xls", "json"]
    for key in preferred_order:
        if key in downloads:
            entry = downloads[key]
            href = None
            if isinstance(entry, dict):
                href = entry.get("href") or entry.get("url")
            elif isinstance(entry, str):
                href = entry
            if href:
                return href, key.upper()

    for value in downloads.values():
        if isinstance(value, dict):
            href = value.get("href") or value.get("url")
            if href:
                return href, "FILE"

    raise ValueError(f"No downloadable file found for version: {version_url}")


def save_downloaded_dataset(dataset_id: str, download_url: str, file_type: str) -> Path:
    dataset_dir = LOCAL_DATA_DIR / dataset_id
    dataset_dir.mkdir(parents=True, exist_ok=True)

    suffix_map = {
        "CSV": ".csv",
        "XLSX": ".xlsx",
        "XLS": ".xls",
        "JSON": ".json",
    }
    suffix = suffix_map.get(file_type, ".data")
    local_path = dataset_dir / f"raw{suffix}"

    time.sleep(0.5)
    response = requests.get(download_url, timeout=120)
    response.raise_for_status()
    local_path.write_bytes(response.content)
    print(f"Saved raw file to: {local_path}")
    return local_path


def load_dataset_from_path(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(file_path, low_memory=False)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(file_path)
    if suffix == ".json":
        with file_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            if "items" in data and isinstance(data["items"], list):
                return pd.DataFrame(data["items"])
            return pd.json_normalize(data)

    try:
        return pd.read_csv(file_path, low_memory=False)
    except Exception:
        pass

    try:
        return pd.read_excel(file_path)
    except Exception:
        pass

    try:
        with file_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            if "items" in data and isinstance(data["items"], list):
                return pd.DataFrame(data["items"])
            return pd.json_normalize(data)
    except Exception:
        pass

    raise ValueError(f"Could not load dataset from file: {file_path}")


def load_dataset_from_url(download_url: str) -> pd.DataFrame:
    try:
        return pd.read_csv(download_url, low_memory=False)
    except Exception:
        pass

    try:
        return pd.read_excel(download_url)
    except Exception:
        pass

    try:
        data = requests.get(download_url, timeout=60).json()
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            if "items" in data and isinstance(data["items"], list):
                return pd.DataFrame(data["items"])
            return pd.json_normalize(data)
    except Exception:
        pass

    raise ValueError(f"Could not load dataset from URL: {download_url}")


def describe_dataframe(dataset_id: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in df.columns:
        s = df[column]
        rows.append(
            {
                "dataset_id": dataset_id,
                "column": column,
                "dtype": str(s.dtype),
                "null_count": int(s.isna().sum()),
                "null_pct": round(float(s.isna().mean() * 100), 2),
                "unique_count": int(s.nunique(dropna=True)),
                "unique_sample": ", ".join(map(str, s.dropna().astype(str).unique()[:5])),
            }
        )
    return pd.DataFrame(rows)


def summarize_categorical(df: pd.DataFrame):
    categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    if not categorical_cols:
        print("No categorical columns detected.")
        return

    print("\nCategorical summary:")
    for col in categorical_cols:
        s = df[col]
        print(f"\n- {col}")
        print(f"  nulls: {s.isna().sum()} | unique values: {s.nunique(dropna=True)}")
        value_counts = s.dropna().value_counts()
        for val, count in value_counts.head(10).items():
            print(f"    {val}: {count}")


In [ ]:
for dataset_id in DATASET_IDS:
    print(f"\n{'=' * 80}")
    print(f"DATASET: {dataset_id}")
    print(f"{'=' * 80}")

    try:
        version_url = get_latest_version_url(dataset_id)
        download_url, file_type = get_download_url(version_url)

        dataset_meta = get_json(f"{ONS_BASE}/datasets/{dataset_id}")
        print("Title:", dataset_meta.get("title"))
        print("Release frequency:", dataset_meta.get("release_frequency"))
        print("Last updated:", dataset_meta.get("last_updated"))
        print("Download format:", file_type)

        raw_path = save_downloaded_dataset(dataset_id, download_url, file_type)
        df = load_dataset_from_path(raw_path)
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print("\nDtypes:")
        print(df.dtypes.to_string())
        print("\nNull counts:")
        print(df.isna().sum().to_string())

        print("\nPreview:")
        print(df.head(3).to_string(index=False))

        summarize_categorical(df)

        profile = describe_dataframe(dataset_id, df)
        profile_path = PROFILE_DIR / f"{dataset_id}_profile.csv"
        profile.to_csv(profile_path, index=False)
        print(f"\nSaved profile to: {profile_path}")
        print(f"Raw file stored at: {raw_path}")

    except Exception as exc:
        print(f"ERROR processing {dataset_id}: {exc}")



DATASET: house-prices-local-authority
Title: House price statistics for small areas in England and Wales
Release frequency: Quarterly
Last updated: 2022-10-14T08:32:49.806Z
Download format: CSV
Saved raw file to: d:\hIntel\Exploration\ONS API\datasets\house-prices-local-authority\raw.csv
Shape: (794400, 14)
Columns: ['V4_1', 'Data Marking', 'calendar-years', 'Time', 'mmm', 'Month', 'administrative-geography', 'Geography', 'property-type', 'PropertyType', 'build-status', 'BuildStatus', 'house-sales-and-prices', 'HouseSalesAndPrices']

Dtypes:
V4_1                        float64
Data Marking                 object
calendar-years                int64
Time                          int64
mmm                          object
Month                        object
administrative-geography     object
Geography                    object
property-type                object
PropertyType                 object
build-status                 object
BuildStatus                  object
house-sales-and-pri